In [7]:
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

PROJECT_ROOT = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}

t0 = time.time()
windows_df = pd.read_parquet(ARTIFACTS_DIR / "section6_windows.parquet")
print(f"✓ Loaded {len(windows_df):,} windows in {time.time()-t0:.2f}s")

# Load SecureBERT tokenizer for token counting
print("Loading SecureBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("ehsanaghaei/SecureBERT")
print("✓ Tokenizer ready")

✓ Loaded 199,990 windows in 3.13s
Loading SecureBERT tokenizer...
✓ Tokenizer ready


/Users/deepakpatnaik/icidea_llm_ids/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
def window_to_text(frames: list) -> str:
    """
    Convert a list of CAN frame dicts into a structured telemetry text string.

    Format per frame:
        [NNN] T=1479121434.850 ID=XXXX DLC=N DATA=XX XX XX XX XX XX XX XX

    - Frame index: 3-digit zero-padded
    - Timestamp: 3 decimal places
    - CAN ID: 4-char zero-padded uppercase hex
    - DLC: integer
    - DATA: DLC bytes as 2-char uppercase hex, space-separated
      (only DLC bytes shown, no zero-padding of payload)
    """
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex  = f"{frame['can_id']:04X}"
        dlc         = frame['dlc']
        data_bytes  = frame['data'][:dlc]   # only show valid bytes
        data_hex    = " ".join(f"{b:02X}" for b in data_bytes)
        ts          = frame['timestamp']
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

In [9]:
t0 = time.time()

texts = []
for _, row in windows_df.iterrows():
    frames = json.loads(row["frames_json"])
    texts.append(window_to_text(frames))

windows_df = windows_df.copy()
windows_df["text"] = texts

print(f"✓ Converted {len(texts):,} windows to text in {time.time()-t0:.1f}s")
print(f"\nSample NORMAL text (first window):")
print("-"*60)
print(texts[0])

✓ Converted 199,990 windows to text in 10.7s

Sample NORMAL text (first window):
------------------------------------------------------------
CAN Bus Telemetry Sequence (14 frames):
[001] T=1479121434.850 ID=0350 DLC=8 DATA=05 28 84 66 6D 00 00 A2
[002] T=1479121434.852 ID=018F DLC=8 DATA=FE 36 00 00 00 3C 00 00
[003] T=1479121434.853 ID=02A0 DLC=8 DATA=60 00 6B 1D 01 04 DD 00
[004] T=1479121434.854 ID=0329 DLC=8 DATA=87 B9 7E 14 12 20 00 14
[005] T=1479121434.855 ID=043F DLC=8 DATA=00 40 60 FF 5A 6C 08 00
[006] T=1479121434.855 ID=0370 DLC=8 DATA=00 20 00 00 00 00 00 00
[007] T=1479121434.860 ID=02C0 DLC=8 DATA=14 00 00 00 00 00 00 00
[008] T=1479121434.863 ID=02A0 DLC=8 DATA=00 00 6B 1D 01 04 DD 00
[009] T=1479121434.865 ID=0370 DLC=8 DATA=00 20 00 00 00 00 00 00
[010] T=1479121434.871 ID=01F1 DLC=8 DATA=00 00 00 00 00 00 00 00
[011] T=1479121434.874 ID=0131 DLC=8 DATA=FB 7F 00 00 20 7F 02 BD
[012] T=1479121434.874 ID=0140 DLC=8 DATA=00 00 00 00 18 00 22 A3
[013] T=1479121434.874 ID=

In [10]:
print("TOKEN COUNT ANALYSIS")
print("="*50)

# Sample 1000 windows per class for token counting
sample_counts = {}
for lbl, name in LABEL_MAP.items():
    class_texts = windows_df[windows_df["label"] == lbl]["text"].sample(
        1000, random_state=42
    ).tolist()
    token_lengths = [
        len(tokenizer.encode(t, add_special_tokens=True))
        for t in class_texts
    ]
    sample_counts[name] = token_lengths
    print(f"  {name:7s}  min={min(token_lengths):3d}  "
          f"max={max(token_lengths):3d}  "
          f"mean={sum(token_lengths)/len(token_lengths):.0f}  "
          f"over_512={sum(1 for x in token_lengths if x > 512):3d}/1000")

all_counts = [c for counts in sample_counts.values() for c in counts]
print(f"\n  Overall  min={min(all_counts):3d}  max={max(all_counts):3d}  "
      f"mean={sum(all_counts)/len(all_counts):.0f}  "
      f"over_512={sum(1 for x in all_counts if x > 512)}/5000")

TOKEN COUNT ANALYSIS
  NORMAL   min=448  max=493  mean=472  over_512=  0/1000


Token indices sequence length is longer than the specified maximum sequence length for this model (516 > 512). Running this sequence through the model will result in indexing errors


  DOS      min=416  max=444  mean=424  over_512=  0/1000
  FUZZY    min=473  max=523  mean=501  over_512=128/1000
  GEAR     min=458  max=486  mean=474  over_512=  0/1000
  RPM      min=430  max=458  mean=448  over_512=  0/1000

  Overall  min=416  max=523  mean=464  over_512=128/5000


In [11]:
# DOS window — should show all ID=0000 with all-zero data
dos_idx = windows_df[windows_df["label"] == 1].index[0]
print("Sample DOS window text:")
print("-"*60)
print(windows_df.loc[dos_idx, "text"])

print()

# FUZZY window — should show varied IDs and random payloads
fuzzy_idx = windows_df[windows_df["label"] == 2].index[0]
print("Sample FUZZY window text:")
print("-"*60)
print(windows_df.loc[fuzzy_idx, "text"])

Sample DOS window text:
------------------------------------------------------------
CAN Bus Telemetry Sequence (14 frames):
[001] T=1478198377.185 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[002] T=1478198377.186 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[003] T=1478198377.188 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[004] T=1478198377.188 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[005] T=1478198377.191 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[006] T=1478198377.196 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[007] T=1478198377.197 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[008] T=1478198377.199 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[009] T=1478198377.199 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[010] T=1478198377.208 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[011] T=1478198377.210 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[012] T=1478198377.210 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[013] T=1478198377.211 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00
[014] T=147819837

In [13]:
print("FILTERING OVER-LENGTH WINDOWS")
print("="*50)

before = len(windows_df)

# Compute token length for every window
print("Tokenizing all windows for length check (takes ~3-4 min)...")
t0 = time.time()
token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=True))
    for text in windows_df["text"]
]
windows_df = windows_df.copy()
windows_df["token_length"] = token_lengths
print(f"  Done in {time.time()-t0:.1f}s")

# Filter
windows_df = windows_df[windows_df["token_length"] <= 512].reset_index(drop=True)
after = len(windows_df)

print(f"\n  Removed {before - after:,} over-length windows")
print(f"  Remaining: {after:,}")
print(f"\nPer-class counts after filtering:")
print(windows_df["label"].value_counts().sort_index().rename(LABEL_MAP).to_string())
print(f"\nToken length stats after filtering:")
print(f"  min={windows_df['token_length'].min()}  "
      f"max={windows_df['token_length'].max()}  "
      f"mean={windows_df['token_length'].mean():.0f}")

FILTERING OVER-LENGTH WINDOWS
Tokenizing all windows for length check (takes ~3-4 min)...
  Done in 77.5s

  Removed 4,921 over-length windows
  Remaining: 195,069

Per-class counts after filtering:
label
NORMAL    39998
DOS       39998
FUZZY     35077
GEAR      39998
RPM       39998

Token length stats after filtering:
  min=416  max=512  mean=462


In [14]:
artifact_path = ARTIFACTS_DIR / "section7_windows_with_text.parquet"
windows_df.drop(columns=["token_length"]).to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6

print(f"✓ Saved {len(windows_df):,} windows to {artifact_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  All windows ≤512 tokens confirmed")
print(f"  Label distribution:")
print(windows_df["label"].value_counts().sort_index().rename(LABEL_MAP).to_string())

✓ Saved 195,069 windows to /Users/deepakpatnaik/icidea_llm_ids/artifacts/section7_windows_with_text.parquet
  Size: 58.7 MB
  All windows ≤512 tokens confirmed
  Label distribution:
label
NORMAL    39998
DOS       39998
FUZZY     35077
GEAR      39998
RPM       39998
